# Relatório do Projeto: Reconhecimento de Sinais de Trânsito GTSRB 
 
Este relatório descreve o trabalho desenvolvido no notebook `Report.ipynb`, explicando o objetivo do projeto, os dados utilizados, as técnicas aplicadas, os modelos testados e os principais resultados obtidos. 
 
O projeto foi desenvolvido no âmbito de Visão por Computador e Processamento de Imagem, com foco na classificação automática de sinais de trânsito usando o conjunto de dados GTSRB (*German Traffic Sign Recognition Benchmark*). 
 
 

## 1. Objetivo 
 
O objetivo principal foi construir e avaliar modelos de Deep Learning capazes de classificar imagens de sinais de trânsito em 43 classes diferentes. 
 
A meta prática do trabalho foi aproximar o desempenho de referência do GTSRB, explorando arquiteturas de redes neuronais convolucionais, técnicas de pré-processamento, balanceamento de dados, aumento de dados, ensemble e transfer learning. 
 
O melhor resultado reportado no desenvolvimento foi de **99.39% de exatidão no conjunto de teste**, obtido com uma arquitetura `RobustCNN` treinada a partir do zero. 
 
 

## 2. Conjunto de Dados Utilizado 
 
Foi utilizado o conjunto de dados **GTSRB**, composto por imagens reais de sinais de trânsito capturadas em diferentes condições de iluminação, escala, perspetiva, nitidez e ruído. 
 
A organização usada segue o formato esperado pelo `torchvision.datasets.ImageFolder`, onde cada classe corresponde a uma pasta com o respetivo identificador numérico. 
 
| Pasta | Função | 
|---|---| 
| `../datasets/original_train_images` | Imagens originais de treino | 
| `../datasets/train_images` | Subconjunto de treino após split | 
| `../datasets/val_images` | Subconjunto de validação | 
| `../datasets/train_balanced` | Treino balanceado com aumento de dados estático | 
| `../datasets/test_images` | Conjunto de teste final | 
 
 

## 3. Bibliotecas e Ferramentas 
 
O projeto foi implementado em Python, usando principalmente: 
 
| Biblioteca | Utilização | 
|---|---| 
| `torch` | Definição, treino e avaliação das redes neuronais | 
| `torchvision` | Transforms, datasets, modelos pré-treinados e aumentos de dados | 
| `numpy` | Operações numéricas auxiliares | 
| `matplotlib` | Visualização de imagens, curvas de treino e gráficos | 
| `pandas` | Organização de matrizes e tabelas | 
| `seaborn` | Visualização de matrizes de confusão | 
| `sklearn.metrics` | Cálculo da matriz de confusão | 
| `OpenCV` / `cv2` | Técnicas clássicas de processamento de imagem, como CLAHE | 
| `PIL` | Leitura e manipulação de imagens | 
| `torchinfo` | Inspeção das arquiteturas | 
 
Foram também usados ficheiros utilitários locais: `util/vcpi_util.py` e `util/our_util.py`. 
 
 

## 4. Preparação dos Dados 
 
A preparação dos dados incluiu quatro passos principais: 
 
1. Carregamento das imagens com `datasets.ImageFolder`. 
2. Redimensionamento das imagens para uma resolução fixa, inicialmente `32x32`. 
3. Conversão para tensor com `transforms.ToTensor()`. 
4. Separação entre treino, validação e teste. 
 
Foi dada atenção especial ao conjunto de validação, porque o GTSRB contém imagens sequenciais. Imagens muito parecidas podem aparecer em frames próximas, por isso o split deve evitar que sequências semelhantes fiquem ao mesmo tempo no treino e na validação. 
 
 

## 5. Análise Exploratória 
 
Antes do treino, o notebook faz uma análise visual e estatística do conjunto de dados: 
 
- Visualização de amostras aleatórias de treino, validação e teste; 
- Verificação da distribuição das classes; 
- Identificação do desbalanceamento do conjunto de dados. 
 
Esta etapa foi importante porque o GTSRB não tem uma distribuição uniforme. Sem correção, o modelo poderia aprender melhor as classes maioritárias e ter pior desempenho nas classes raras. 
 
 

## 6. Balanceamento e Aumento de Dados 
 
Para lidar com o desbalanceamento, foi criado um conjunto `train_balanced`, onde classes minoritárias foram aumentadas até um número alvo de imagens. 
 
Foram usadas técnicas de aumento de dados estático, isto é, novas imagens foram criadas e gravadas em disco antes do treino. 
 
Transformações usadas ou testadas: 
 
- Rotações pequenas; 
- Translações; 
- Alteração de escala; 
- Shear; 
- Alteração de brilho, contraste e saturação; 
- Perspetiva; 
- Blur; 
- Ruído suave; 
- Sharpness e grayscale como opções experimentais. 
 
Também foram testadas estratégias de **aumento de dados dinâmico**, aplicadas durante o treino com `torchvision.transforms`. 
 
 

## 7. Modelo Baseline 
 
O primeiro modelo implementado foi uma CNN simples, usada como referência experimental. 
 
A `BaselineCNN` contém: 
 
- Duas camadas convolucionais; 
- Ativações `ReLU`; 
- `MaxPool2d` para redução espacial; 
- Uma camada totalmente ligada final. 
 
Este modelo serviu para estabelecer um ponto de partida. O desempenho reportado para o baseline foi aproximadamente **87.12% no conjunto de teste** na conclusão consolidada do notebook. 
 
 

## 8. Infraestrutura de Treino 
 
O treino foi feito com uma função genérica `train_model`, responsável por: 
 
- Colocar o modelo no dispositivo correto (`CPU` ou `GPU`); 
- Executar forward pass e backward pass; 
- Atualizar os pesos com o otimizador; 
- Calcular loss e exatidão de treino; 
- Avaliar em validação após cada época; 
- Aplicar scheduler de taxa de aprendizagem; 
- Guardar automaticamente o melhor checkpoint com base na perda de validação; 
- Parar cedo com `Early_Stopping` quando a validação deixa de melhorar. 
 
Foram usados `CrossEntropyLoss`, `Adam`, `ReduceLROnPlateau` e `Early_Stopping`. 
 
 

## 9. RobustCNN 
 
A principal melhoria do projeto foi a substituição da baseline por uma arquitetura mais robusta, chamada `RobustCNN`. 
 
A arquitetura inclui: 
 
- Yrês blocos convolucionais; 
- Maior número de canais (`32`, `64`, `128`); 
- `BatchNorm2d` após convoluções; 
- Ativações `ReLU`; 
- `MaxPool2d` para redução espacial; 
- `Dropout` para regularização; 
- **Global Average Pooling**, reduzindo a dependência de uma camada `Flatten` grande. 
 
Esta alteração foi a que trouxe o maior ganho do projeto. Segundo a conclusão do notebook, a passagem de `BaselineCNN` para `RobustCNN` elevou o resultado de **87.12% para 99.39%** no conjunto de teste. 
 
 

## 10. Experiências com Aumento de Dados Dinâmico 
 
Depois da `RobustCNN`, foram testadas várias variantes de aumento de dados dinâmico: 
 
| Estratégia | Ideia | 
|---|---| 
| Apenas estático | Treino com conjunto de dados já balanceado em disco | 
| Estático + dinâmico | Conjunto de dados balanceado + transforms aleatórias durante treino | 
| Apenas dinâmico | Conjunto de dados com aumento de dados em tempo de treino sem aumentar ficheiros | 
| Dinâmico massivo | Concatenação de múltiplas versões dinâmicas do conjunto de dados | 
 
Apesar de algumas destas estratégias melhorarem a exatidão de validação, elas não superaram de forma consistente a `RobustCNN` estática no conjunto de teste. 
 
 

## 11. Processamento Clássico de Imagem 
 
Também foram testadas técnicas clássicas de processamento de imagem para tentar melhorar a robustez visual dos sinais. 
 
Foram exploradas: 
 
- **CLAHE** no canal de luminância em espaço LAB; 
- Normalização por média e desvio-padrão do conjunto de dados; 
- Gamma correction para simular variações de iluminação; 
- Combinação de CLAHE com aumento de dados dinâmico. 
 
Estas técnicas são teoricamente adequadas para lidar com sombras e iluminação irregular. No entanto, no notebook, as experiências com CLAHE e normalização não superaram claramente o pipeline mais simples com `RobustCNN`. 
 
 

## 12. Experiência com Resolução 
 
O projeto também testou a diferença entre imagens `32x32` e `48x48`. 
 
A hipótese era que `48x48` poderia preservar mais detalhe visual dos sinais. Contudo, aumentar a resolução também aumenta o custo computacional e pode exigir mais regularização. 
 
No desenvolvimento registado, a resolução maior não produziu uma melhoria clara suficiente para substituir o melhor resultado obtido com o setup mais simples. 
 
 

## 13. Ensembles 
 
Foram testados ensembles com vários checkpoints treinados em condições diferentes. 
 
Duas estratégias foram usadas: 
 
- **Soft voting**: soma dos logits antes do `argmax`. 
- **Hard voting**: votação por classe prevista. 
 
O objetivo era verificar se modelos treinados com pipelines diferentes cometiam erros complementares. No entanto, o ensemble não superou claramente a melhor `RobustCNN` individual tendo atingido unicamente uma accuracy de 99.22%. 
 
 

## 14. Transfer Learning 
 
Foram também testados modelos pré-treinados no ImageNet: 
 
- `ResNet-18`; 
- `EfficientNet-B0`. 
 
A estratégia incluiu uma fase inicial de treino apenas da cabeça de classificação e depois fine-tuning completo. 
 
Apesar de serem arquiteturas fortes, os resultados reportados ficaram abaixo da melhor `RobustCNN` treinada do zero. Isto é plausível porque o GTSRB é um problema específico, com imagens pequenas e classes bem definidas. 
 
 

## 15. Test-Time Augmentation 
 
Foi adicionada uma melhoria adicional ao notebook: **Test-Time Augmentation (TTA)**. 
 
A TTA não altera o treino nem os pesos do modelo. Durante a avaliação, cada imagem de teste é avaliada em várias versões ligeiramente modificadas, e os logits são combinados antes da decisão final. 
 
As transformações usadas foram conservadoras: 
 
- Brilho ligeiramente maior; 
- Brilho ligeiramente menor; 
- Contraste leve; 
- Deslocamentos de 1 pixel em quatro direções. 
 
Não foram usados flips horizontais porque, em sinais de trânsito, uma inversão pode mudar o significado visual ou criar uma imagem pouco realista. 
 
 

## 16. Resultados Principais 
 
| Modelo / Estratégia | Exatidão no teste reportada | 
|---|---:| 
| BaselineCNN | 87.12% | 
| RobustCNN estática | **99.39%** | 
| RobustCNN estática + dinâmica | 99.00% | 
| RobustCNN apenas dinâmica | 99.07% | 
| RobustCNN dinâmica massiva | 98.80% | 
| CLAHE + normalização | 97.51% | 
| CLAHE + dinâmico | 98.82% | 
| ResNet-18 Transfer Learning | 98.46% | 
| EfficientNet-B0 Transfer Learning | 98.24% | 
| Ensemble soft voting | ~99.22% - 99.28% | 
| Ensemble hard voting | ~99.22% - 99.25% | 
 
O melhor resultado identificado foi a `RobustCNN` treinada a partir do zero com dados estaticamente balanceados. 
 
 

## 17. Interpretação dos Resultados 
 
A conclusão principal é que a arquitetura teve mais impacto do que as restantes técnicas. 
 
A `BaselineCNN` tinha capacidade limitada, o que se refletiu num desempenho claramente inferior. A `RobustCNN`, ao adicionar mais profundidade, batch normalization, dropout e global average pooling, conseguiu generalizar muito melhor. 
 
As técnicas adicionais, como aumento de dados dinâmico, CLAHE, ensembles e transfer learning, foram úteis para exploração experimental, mas não ultrapassaram consistentemente o modelo principal. 
 
 

## 18. Pontos Fortes do Trabalho 
 
O trabalho apresenta vários aspetos positivos: 
 
- Pipeline completo de treino, validação e teste; 
- Análise da distribuição das classes; 
- Criação de conjunto de dados balanceado; 
- Comparação entre baseline e modelo melhorado; 
- Uso de early stopping e checkpointing; 
- Avaliação com matrizes de confusão; 
- Comparação com transfer learning; 
- Exploração de técnicas clássicas de processamento de imagem; 
- Teste de ensembles; 
- Adição de TTA para avaliação mais robusta. 
 
 

## 19. Melhorias Futuras 
 
Como trabalho futuro, o grupo identificou que, poderia ser importante tornar a avaliação ainda mais rigorosa e reprodutível. Em particular, poderiam ser acrescentadas as seguintes alterações: 
 
- Fixar seeds para reduzir variação entre execuções; 
- Calcular precision, recall e F1-score por classe; 
- Guardar automaticamente os resultados de cada experiência numa tabela ou ficheiro CSV; 
- Analisar visualmente todas as imagens mal classificadas; 
- Comparar o modelo com e sem TTA numa execução limpa; 
- Testar `AdamW`, `label_smoothing` e schedulers alternativos; 
- Construir ensembles ponderados apenas com os modelos mais fortes. 
 
 

## 20. Conclusão 
 
O grupo desenvolveu durante este trabalho um pipeline completo para classificação de sinais de trânsito no conjunto de dados GTSRB. A evolução principal foi a passagem de uma CNN simples para uma arquitetura `RobustCNN`, que aumentou significativamente a capacidade de representação e reduziu overfitting através de batch normalization, dropout e global average pooling. 
 
O melhor resultado reportado foi **99.39% de exatidão no conjunto de teste**, valor este que pode ser considerado elevado para o problema, mas ainda relativamente distante dos valores obtidos com modelos _state of the art_. As experiências adicionais com aumento de dados, CLAHE, ensembles e transfer learning contribuíram para validar escolhas de projeto, mesmo quando não superaram o melhor modelo. 
 
Assim, o trabalho demonstra não só a construção de um modelo de alto desempenho, mas também uma abordagem experimental comparativa, onde diferentes técnicas foram testadas e avaliadas de forma crítica. 
 
 